# MCP Demo: From Tool Calling to the Model Context Protocol

In the tool calling demo, the `get_weather` function, its tool definition and the code that ran it all lived **inside the notebook**. In this demo we move the tool into its own **MCP server**, and the notebook connects to it as an **MCP client**.

```
This Colab machine
┌──────────────────────────────────────┐        ┌──────────────┐
│  This notebook = HOST + MCP CLIENT   │ ─────▶ │  OpenAI LLM  │
│               ▲                      │  API   │  (cloud)     │
│               │ stdio                │        └──────────────┘
│               ▼                      │
│  weather_server.py = MCP SERVER      │
└──────────────────────────────────────┘
```

The server runs on the **same machine** as the notebook, so it is a **local** server, connected over **stdio**. (In Colab, "the same machine" is Google's cloud machine, not your laptop.)

## Learning objectives

1. **Build** an MCP server from an ordinary Python function.
2. **Discover** its tools from a client, and compare the result with a hand-written tool definition.
3. **Call** a tool directly, without any LLM.
4. **Connect** MCP to the tool-calling agent loop.
5. **Extend** the server with a new tool, without changing the client.

## 0. Setup

We pin the MCP SDK version, because its API changed in version 2. Many tutorials online still show the older `FastMCP` style.

In [1]:
!pip install -q "mcp==2.2.0" openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.5 MB/s eta 0:00:00


Add your OpenAI API key as a Colab secret named `OPENAI_API_KEY` (key icon in the left sidebar, with **Notebook access** turned on). If that fails, the next cell asks you to type it.

In [2]:
import json
import os
import sys
from contextlib import asynccontextmanager

from openai import OpenAI


def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")


os.environ["OPENAI_API_KEY"] = load_openai_key()
openai_client = OpenAI()
MODEL = "gpt-4o-mini"
print("OpenAI client is ready. Model:", MODEL)

OpenAI client is ready. Model: gpt-4o-mini


## 1. Build the MCP server

This is the **same `get_weather` function** as in the tool calling demo, with the same simulated data. Only three things are new:

- `mcp = MCPServer("weather")` creates the server.
- `@mcp.tool()` turns the function into an MCP tool. The SDK builds the tool definition from the **function name**, the **docstring** and the **type hints**. `Annotated[..., Field(description=...)]` adds a description to an argument.
- `mcp.run(transport="stdio")` serves it over stdio.

The cell below saves the server as a file. It doesn't run it: the client starts it later.

In [15]:
%%writefile weather_server.py
"""weather_server.py: an MCP server with simulated weather data (version 1)."""
import logging
from typing import Annotated

from pydantic import Field
from mcp.server import MCPServer

# A stdio server must NEVER print() to stdout: stdout carries the MCP messages.
# Use logging instead, which wra
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("weather_server")

mcp = MCPServer("weather")

WEATHER_DATA = {
    "Singapore": {"temperature": 31, "condition": "Partly cloudy", "humidity": 75},
    "London": {"temperature": 18, "condition": "Rainy", "humidity": 80},
    "Tokyo": {"temperature": 27, "condition": "Sunny", "humidity": 65},
}


@mcp.tool()
def get_weather(
    location: Annotated[str, Field(description="The city name, for example Singapore, London, or Tokyo.")],
) -> dict:
    """Get the current weather for a specified city. Temperatures are in Celsius."""
    logger.info("get_weather(%s)", location)
    if location in WEATHER_DATA:
        return {"location": location, **WEATHER_DATA[location]}
    return {"location": location, "error": "Weather data is not available for this location."}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server.py


### Connect to the server

`connect()` is a small helper used in the rest of the notebook. It:

1. describes how to **launch the server**: run `python weather_server.py` as a subprocess;
2. opens an MCP **client** over stdio;
3. sends the server's log messages (its stderr) to `server.log`.

Leaving the `async with` block disconnects and stops the server process.

> In Colab you can use `await` directly in a cell. In a normal `.py` file, you would wrap this code in a function and run it with `asyncio.run(...)`.

In [16]:
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["weather_server.py"])


@asynccontextmanager
async def connect():
    """Start the weather server as a subprocess and connect an MCP client to it."""
    with open("server.log", "a") as server_log:
        async with Client(stdio_client(SERVER, errlog=server_log)) as client:
            yield client

## 2. Discover the tools

The client asks the server which tools it offers (`tools/list`). Nothing about the tools is written in the notebook: everything below comes from the server.

In [17]:
async with connect() as client:
    result = await client.list_tools()

for tool in result.tools:
    print("Name:       ", tool.name)
    print("Description:", tool.description)
    print("Input schema:")
    print(json.dumps(tool.input_schema, indent=2))

Name:        get_weather
Description: Get the current weather for a specified city. Temperatures are in Celsius.
Input schema:
{
  "type": "object",
  "properties": {
    "location": {
      "description": "The city name, for example Singapore, London, or Tokyo.",
      "title": "Location",
      "type": "string"
    }
  },
  "required": [
    "location"
  ],
  "title": "get_weatherArguments"
}


**Compare** this with the tool definition we wrote **by hand** in the tool calling demo:

```python
"parameters": {
    "type": "object",
    "properties": {
        "location": {
            "type": "string",
            "description": "The city name, for example Singapore, London, or Tokyo.",
        }
    },
    "required": ["location"],
    ...
}
```

The `input_schema` is the same **JSON Schema**, generated from the type hints. The name and description come from the function name and docstring. So an MCP server's tool list contains everything the LLM needs: name, description and argument schema.

## 3. Call a tool directly (no LLM)

MCP is a protocol between a **client** and a **server**. An LLM is optional. Here our code calls the tool itself (`tools/call`).

In [18]:
async with connect() as client:
    result = await client.call_tool("get_weather", {"location": "Tokyo"})

print("Is error:", result.is_error)
print(result.content[0].text)

Is error: False
{
  "location": "Tokyo",
  "temperature": 27,
  "condition": "Sunny",
  "humidity": 65
}


The result is a list of **content blocks** (here, one text block containing the dictionary as JSON), plus an `is_error` flag.

**What happens with bad input?** The server checks the arguments against its schema, so it rejects a number where a string is expected. The error comes back as a normal result with `is_error=True`, rather than crashing the client.

In [19]:
async with connect() as client:
    result = await client.call_tool("get_weather", {"location": 42})

print("Is error:", result.is_error)
print(result.content[0].text)

Is error: True
Error executing tool get_weather: 1 validation error for get_weatherArguments
location
  Input should be a valid string [type=string_type, input_value=42, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type


## 4. Agent + MCP

Now we connect MCP to the **same agent loop** as in the tool calling demo. Only **two things change** (marked `# MCP:` in the code):

1. **Where the tool definitions come from:** we no longer write them by hand. We ask the server (`list_tools`) and convert them into OpenAI's Chat Completions format.
2. **Who runs the tool:** we no longer call a local Python function. We send the call to the server (`call_tool`).

Everything else (sending messages, reading `tool_calls`, returning `"role": "tool"` messages, `MAX_TURNS`) is unchanged.

These are the steps on the slide *How MCP Plugs Into Tool Calling*: discover → convert → decide → execute → answer.

In [21]:
def to_openai_tools(mcp_tools):
    """Convert MCP tool definitions into OpenAI Chat Completions tool definitions."""
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,  # the same JSON Schema
            },
        }
        for tool in mcp_tools
    ]
    # Note: we don't set "strict": True here. OpenAI's strict mode needs
    # "additionalProperties": False on every object, which MCP schemas don't include.


def result_to_text(result):
    """Join the text blocks of an MCP tool result into one string for the LLM."""
    text = "\n".join(block.text for block in result.content if block.type == "text")
    return f"Error: {text}" if result.is_error else text


MAX_TURNS = 5


async def run_agent(user_question):
    async with connect() as client:
        # MCP (1): discover the tools from the server and convert them for OpenAI
        mcp_tools = (await client.list_tools()).tools
        tools = to_openai_tools(mcp_tools)
        print("Tools from the MCP server:", [t.name for t in mcp_tools])

        messages = [{"role": "user", "content": user_question}]

        for turn in range(MAX_TURNS):
            response = openai_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
            )
            message = response.choices[0].message
            messages.append(message)

            # No tool call -> final answer
            if not message.tool_calls:
                print("\nFinal answer:")
                print(message.content)
                return message.content

            # Execute the requested tools
            for tool_call in message.tool_calls:
                name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                print(f"\nLLM requested tool: {name}({arguments})")

                # MCP (2): the MCP server runs the tool, not a local function
                result = await client.call_tool(name, arguments)
                output = result_to_text(result)
                print("Tool result:", output)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": output,
                    }
                )

        print(f"\nStopped after {MAX_TURNS} turns without a final answer.")
        return None

In [23]:
await run_agent("What's the weather like in Singapore?")

Tools from the MCP server: ['get_weather']

LLM requested tool: get_weather({'location': 'Singapore'})
Tool result: {
  "location": "Singapore",
  "temperature": 31,
  "condition": "Partly cloudy",
  "humidity": 75
}

Final answer:
The weather in Singapore is currently partly cloudy with a temperature of 31°C and humidity at 75%.


'The weather in Singapore is currently partly cloudy with a temperature of 31°C and humidity at 75%.'

## 5. Add a new tool without changing the client

This is the main benefit of MCP. We add a second tool, `convert_temperature`, **to the server only**, then run the **same** `run_agent()` again. The client code isn't touched.

In [24]:
%%writefile weather_server.py
"""weather_server.py: an MCP server with simulated weather data (version 2)."""
import logging
from typing import Annotated

from pydantic import Field
from mcp.server import MCPServer

# A stdio server must NEVER print() to stdout: stdout carries the MCP messages.
# Use logging instead, which writes to stderr.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("weather_server")

mcp = MCPServer("weather")

WEATHER_DATA = {
    "Singapore": {"temperature": 31, "condition": "Partly cloudy", "humidity": 75},
    "London": {"temperature": 18, "condition": "Rainy", "humidity": 80},
    "Tokyo": {"temperature": 27, "condition": "Sunny", "humidity": 65},
}


@mcp.tool()
def get_weather(
    location: Annotated[str, Field(description="The city name, for example Singapore, London, or Tokyo.")],
) -> dict:
    """Get the current weather for a specified city. Temperatures are in Celsius."""
    logger.info("get_weather(%s)", location)
    if location in WEATHER_DATA:
        return {"location": location, **WEATHER_DATA[location]}
    return {"location": location, "error": "Weather data is not available for this location."}

@mcp.tool()
def convert_temperature(
    celsius: Annotated[float, Field(description="Temperature in degrees Celsius.")],
) -> dict:
    """Convert a temperature from Celsius to Fahrenheit."""
    logger.info("convert_temperature(%s)", celsius)
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, 1)}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server.py


In [25]:
await run_agent("What's the weather in London, in Fahrenheit?")

Tools from the MCP server: ['get_weather', 'convert_temperature']

LLM requested tool: get_weather({'location': 'London'})
Tool result: {
  "location": "London",
  "temperature": 18,
  "condition": "Rainy",
  "humidity": 80
}

LLM requested tool: convert_temperature({'celsius': 18})
Tool result: {
  "celsius": 18.0,
  "fahrenheit": 64.4
}

Final answer:
The current weather in London is rainy, with a temperature of 64.4°F.


'The current weather in London is rainy, with a temperature of 64.4°F.'

The agent found the new tool through discovery and **chained** the two: `get_weather` first, then `convert_temperature` with the result. That took more than one turn of the loop.

**Takeaway:** the tools live in the server and the app only connects to it. Any MCP-compatible app (Claude Desktop, an IDE, another agent) could use this same server without changes.

### Where did the server's logs go?

The server's `logger.info(...)` messages went to **stderr**, which `connect()` saved in `server.log`. Its stdout was reserved for MCP messages.

In [26]:
print(open("server.log").read()[-1500:])

INFO:weather_server:get_weather(Tokyo)
INFO:mcp.server.mcpserver.server:Tool 'get_weather' rejected arguments: ['location']
INFO:weather_server:get_weather(Singapore)
INFO:weather_server:get_weather(Tokyo)
INFO:mcp.server.mcpserver.server:Tool 'get_weather' rejected arguments: ['location']
INFO:weather_server:get_weather(Singapore)
INFO:weather_server:get_weather(Singapore)
INFO:weather_server:get_weather(London)
INFO:weather_server:convert_temperature(18.0)



## 6. (Optional) Resources and prompts

Tools are one of three things an MCP server can expose:

| | Chosen by | Example below |
|---|---|---|
| **Tools** | the model | `get_weather`, `convert_temperature` |
| **Resources** | the application | `weather://cities`, a list of supported cities |
| **Prompts** | the user | `travel_briefing`, a reusable prompt template |

In [49]:
%%writefile weather_server.py
"""weather_server.py: an MCP server with simulated weather data (version 3)."""
import logging
from typing import Annotated

from pydantic import Field
from mcp.server import MCPServer

# A stdio server must NEVER print() to stdout: stdout carries the MCP messages.
# Use logging instead, which writes to stderr.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("weather_server")

mcp = MCPServer("weather")

WEATHER_DATA = {
    "Singapore": {"temperature": 31, "condition": "Partly cloudy", "humidity": 75},
    "London": {"temperature": 18, "condition": "Rainy", "humidity": 80},
    "Tokyo": {"temperature": 27, "condition": "Sunny", "humidity": 65},
}


@mcp.tool()
def get_weather(
    location: Annotated[str, Field(description="The city name, for example Singapore, London, or Tokyo.")],
) -> dict:
    """Get the current weather for a specified city. Temperatures are in Celsius."""
    logger.info("get_weather(%s)", location)
    if location in WEATHER_DATA:
        return {"location": location, **WEATHER_DATA[location]}
    return {"location": location, "error": "Weather data is not available for this location."}

@mcp.tool()
def convert_temperature(
    celsius: Annotated[float, Field(description="Temperature in degrees Celsius.")],
) -> dict:
    """Convert a temperature from Celsius to Fahrenheit."""
    logger.info("convert_temperature(%s)", celsius)
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, 1)}

@mcp.resource("weather://cities")
def supported_cities() -> str:
    """The list of cities this server has weather data for."""
    return ", ".join(WEATHER_DATA)


@mcp.prompt()
def travel_briefing(city: str) -> str:
    """A reusable prompt for a short travel weather briefing."""
    return (f"Check the current weather in {city} and write a 3-line travel briefing: "
            "what to wear, what to bring, and one tip.")


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server.py


In [50]:
async with connect() as client:
    # Resources: data the application can read, identified by a URI
    resources = (await client.list_resources()).resources
    print("Resources:", [(r.uri, r.description) for r in resources])
    cities = await client.read_resource("weather://cities")
    print("weather://cities ->", cities.contents[0].text)

    # Prompts: templates the user can pick, often shown as slash commands
    prompts = (await client.list_prompts()).prompts
    print("\nPrompts:", [(p.name, [a.name for a in (p.arguments or [])]) for p in prompts])
    briefing = await client.get_prompt("travel_briefing", {"city": "Tokyo"})
    prompt_text = briefing.messages[0].content.text
    print("travel_briefing(Tokyo) ->", prompt_text)

Resources: [('weather://cities', 'The list of cities this server has weather data for.')]
weather://cities -> Singapore, London, Tokyo

Prompts: [('travel_briefing', ['city'])]
travel_briefing(Tokyo) -> Check the current weather in Tokyo and write a 3-line travel briefing: what to wear, what to bring, and one tip.


A user-chosen prompt is just a ready-made request. Here we send it to the agent:

In [39]:
await run_agent(prompt_text)

Tools from the MCP server: ['get_weather', 'convert_temperature']

LLM requested tool: get_weather({'location': 'Tokyo'})
Tool result: {
  "location": "Tokyo",
  "temperature": 27,
  "condition": "Sunny",
  "humidity": 65
}

LLM requested tool: convert_temperature({'celsius': 27})
Tool result: {
  "celsius": 27.0,
  "fahrenheit": 80.6
}

Final answer:
**Travel Briefing for Tokyo:**

- **What to Wear:** Dress in light, breathable clothing since it's sunny and around 27°C (80.6°F).
- **What to Bring:** Don't forget sunscreen and a hat to protect against the sun, and stay hydrated with water.
- **Tip:** Use public transportation; it's convenient and an excellent way to explore the city's unique districts!


"**Travel Briefing for Tokyo:**\n\n- **What to Wear:** Dress in light, breathable clothing since it's sunny and around 27°C (80.6°F).\n- **What to Bring:** Don't forget sunscreen and a hat to protect against the sun, and stay hydrated with water.\n- **Tip:** Use public transportation; it's convenient and an excellent way to explore the city's unique districts!"

## 7. Exercises

1. Ask about a city the server doesn't know, such as `Paris`. How does the agent use the error result?
2. Ask *"Compare the weather in Singapore and Tokyo."* How many tool calls does the model make?
3. Add a third tool, for example `get_clothing_advice(temperature: float)`, to the server cell in section 6 (the latest version). Run that cell, then call `run_agent()` again **without changing any client code**.



## Summary

| | Tool calling demo | MCP demo |
|---|---|---|
| Tool code lives in | the notebook | a separate server process |
| Tool definition | written by hand | generated by the server, discovered with `list_tools` |
| Running a tool | call a local function | `call_tool` sends it to the server |
| Adding a tool | change the app | change the server only |
| The agent loop | the same | the same |

In [ ]:
#Solution
"""
@mcp.tool()
def get_clothing_advice(
    temperature: Annotated[float, Field(description="Temperature in degrees Celsius.")],
) -> dict:
    """Suggest what to wear for a given temperature in degrees Celsius."""
    logger.info("get_clothing_advice(%s)", temperature)
    if temperature >= 30:
        advice = "Light, breathable clothes such as a t-shirt and shorts. Bring water and sunscreen."
    elif temperature >= 22:
        advice = "T-shirt and light trousers or a skirt."
    elif temperature >= 15:
        advice = "Long sleeves or a light jacket."
    elif temperature >= 5:
        advice = "A warm coat, plus a scarf if it's windy."
    else:
        advice = "A heavy coat, gloves, a hat and a scarf."
    return {"temperature": temperature, "advice": advice}
  """